In [ ]:
#CNN implementation here
#using the LOSO method, same as the random forest, so results are directly comparable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
#sliding window for CNN — returns raw sequences instead of statistical features
#window shape: (n_windows, window_size, n_features) so the CNN sees the full time series
def sliding_window_sequences(df, window_size=100, step=50):
    feature_cols = df.columns.drop("activityID")
    X, y = [], []

    for start in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start:start + window_size]

        label = window["activityID"].mode()[0]

        X.append(window[feature_cols].values)   # shape: (100, 36)
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y)

In [ ]:
import os
os.makedirs(r"/content/CPEN355_FinalProject/preprocessing/data", exist_ok=True)

for i in range(1, 9):
    globals()[f"data_{i}"] = pd.read_pickle(rf"/content/CPEN355_FinalProject/preprocessing/data/data_{i}.pkl")

X_all = {}
y_all = {}

for i in range(1, 9):
    df = globals()[f"data_{i}"]
    X, y = sliding_window_sequences(df, window_size=100, step=50)
    X_all[i] = X
    y_all[i] = y

#fit a single label encoder across all subjects so class indices are consistent
all_labels = np.concatenate(list(y_all.values()))
le = LabelEncoder()
le.fit(all_labels)
for i in range(1, 9):
    y_all[i] = le.transform(y_all[i])

In [ ]:
class MultiScaleBlock(nn.Module):
    """Parallel convolutions at 3 different timescales, then concatenate."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        branch_ch = out_ch // 3  # split output channels across branches

        self.branch_short  = nn.Sequential(
            nn.Conv1d(in_ch, branch_ch, kernel_size=3,  padding=1),
            nn.InstanceNorm1d(branch_ch, affine=True), nn.ReLU()
        )
        self.branch_medium = nn.Sequential(
            nn.Conv1d(in_ch, branch_ch, kernel_size=7,  padding=3),
            nn.InstanceNorm1d(branch_ch, affine=True), nn.ReLU()
        )
        self.branch_long   = nn.Sequential(
            nn.Conv1d(in_ch, branch_ch, kernel_size=15, padding=7),
            nn.InstanceNorm1d(branch_ch, affine=True), nn.ReLU()
        )

    def forward(self, x):
        return torch.cat([
            self.branch_short(x),
            self.branch_medium(x),
            self.branch_long(x)
        ], dim=1)   # concatenate along channel dim


class PAMAP2_CNN(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()

        self.conv_block = nn.Sequential(
            # Block 1: multi-scale (output: 3 × 21 = 63 channels, close to 64)
            MultiScaleBlock(n_features, 63),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.2),

            # Block 2: multi-scale
            MultiScaleBlock(63, 126),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.2),

            # Block 3: single scale to compress
            nn.Conv1d(126, 256, kernel_size=3, padding=1),
            nn.InstanceNorm1d(256, affine=True),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))

In [ ]:
#LOSO cross-validation — mirrors the random forest loop exactly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

N_FEATURES  = X_all[1].shape[2]   # 36
N_CLASSES   = len(le.classes_)    # 8
EPOCHS      = 50
BATCH_SIZE  = 64
LR          = 1e-4

results = {}

for test_subject in range(1, 9):
    #build train arrays
    x_train = np.vstack([X_all[i] for i in range(1, 9) if i != test_subject])
    y_train = np.concatenate([y_all[i] for i in range(1, 9) if i != test_subject])
    x_test  = X_all[test_subject]
    y_test  = y_all[test_subject]

    #normalise per feature — fit on train, apply to test
    n_train, W, F = x_train.shape
    n_test        = x_test.shape[0]
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train.reshape(-1, F)).reshape(n_train, W, F)
    x_test  = scaler.transform(x_test.reshape(-1, F)).reshape(n_test, W, F)

    #CNN expects (N, channels, length) — transpose from (N, W, F) to (N, F, W)
    x_train_t = torch.tensor(x_train.transpose(0, 2, 1))
    x_test_t  = torch.tensor(x_test.transpose(0, 2, 1))
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t  = torch.tensor(y_test,  dtype=torch.long)

    train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TensorDataset(x_test_t,  y_test_t),  batch_size=BATCH_SIZE)

    #fresh model for each fold
    model     = PAMAP2_CNN(N_FEATURES, N_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(EPOCHS):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

    #evaluate
    model.eval()
    all_preds = []
    with torch.no_grad():
        for Xb, _ in test_loader:
            all_preds.extend(model(Xb.to(device)).argmax(1).cpu().numpy())

    acc = accuracy_score(y_test, all_preds)
    results[test_subject] = {"accuracy": acc, "report": classification_report(y_test, all_preds, target_names=le.classes_.astype(str))}

    print(f"Subject {test_subject} (test) — Accuracy: {acc:.4f}")


print(f"\nMean LOSO Accuracy: {np.mean([r['accuracy'] for r in results.values()]):.4f}")

In [ ]:
subjects = list(results.keys())
acc      = [results[s]['accuracy'] for s in subjects]

plt.figure(figsize=(8, 6))
plt.bar(subjects, acc, color='steelblue')
plt.axhline(y=np.mean(acc), color='red', linestyle='--', label=f'Mean: {np.mean(acc):.4f}')
plt.xlabel('Subject')
plt.ylabel('Accuracy')
plt.title('LOSO Accuracy per Subject (CNN)')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('loso_accuracy_cnn.png', dpi=150)
plt.show()